In [ ]:
# ==============================================================
# AUDIO CLASSIFICATION + SPEECH-TO-TEXT + TEXT-TO-SPEECH
# Dataset: ESC-50 Environmental Sound Classification
# Deep Learning: PyTorch MLP
# Audio Features: MFCC
# Speech Recognition: Whisper Tiny
# Text-to-Speech: gTTS
#
# Works in Google Colab and local Jupyter Notebook.
# The first run downloads the ESC-50 dataset.
# ==============================================================


# STEP 1: INSTALL THE REQUIRED LIBRARIES
# Run this cell first. If using a local Jupyter notebook,
# restart the kernel if the installation requests it.

import sys
import subprocess

packages = [
    "numpy", "pandas", "matplotlib", "seaborn", "librosa",
    "soundfile", "scikit-learn", "torch", "transformers",
    "gTTS", "joblib", "requests"
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages
)


# STEP 2: IMPORT LIBRARIES

import os
import random
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import torch
import torch.nn as nn
import joblib

from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from IPython.display import Audio, display

# Set a random seed so that the results are more reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use the GPU automatically if one is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)


# STEP 3: DOWNLOAD THE ESC-50 DATASET
# ESC-50 contains 2,000 environmental sound recordings
# organised into 50 different sound categories.

dataset_folder = Path("ESC-50-master")
zip_path = Path("ESC-50.zip")

if not (dataset_folder / "meta" / "esc50.csv").exists():

    print("Downloading the ESC-50 dataset...")

    url = (
        "https://github.com/karolpiczak/ESC-50/"
        "archive/refs/heads/master.zip"
    )

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with open(zip_path, "wb") as file:
        file.write(response.content)

    print("Extracting dataset...")

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(".")

    print("Dataset downloaded successfully!")

else:
    print("ESC-50 dataset already exists.")


# STEP 4: LOAD THE DATASET INFORMATION
# The CSV file tells us each audio filename and its sound category.

metadata_path = dataset_folder / "meta" / "esc50.csv"
audio_folder = dataset_folder / "audio"

metadata = pd.read_csv(metadata_path)

print("\nDataset preview:")
print(metadata.head())

print("\nTotal audio recordings:", len(metadata))
print("Total sound categories:", metadata["category"].nunique())

print("\nAvailable categories:")
print(sorted(metadata["category"].unique()))


# STEP 5: SELECT A MANAGEABLE SUBSET
# To keep the first training run quicker on a laptop,
# we use up to 1,000 recordings, with similar numbers per class.
# Set MAX_SAMPLES = None if you want to use the complete dataset.

MAX_SAMPLES = 1000

if MAX_SAMPLES is not None and MAX_SAMPLES < len(metadata):

    samples_per_class = max(
        1, MAX_SAMPLES // metadata["category"].nunique()
    )

    metadata = (
        metadata.groupby("category", group_keys=False)
        .sample(n=samples_per_class, random_state=SEED)
        .reset_index(drop=True)
    )

print("\nRecordings selected for this run:", len(metadata))


# STEP 6: EXTRACT MFCC FEATURES FROM THE AUDIO
# MFCC features describe the sound's frequency characteristics.
# We calculate the mean and standard deviation of 40 MFCCs.
# This gives each audio clip a fixed-size vector of 80 values.

def extract_mfcc(audio_path):
    signal, sample_rate = librosa.load(
        audio_path,
        sr=22050,
        mono=True
    )

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sample_rate,
        n_mfcc=40
    )

    # Combining mean and standard deviation captures
    # the overall sound characteristics in a compact form.
    mean_features = np.mean(mfcc, axis=1)
    std_features = np.std(mfcc, axis=1)

    return np.concatenate([mean_features, std_features])


features = []
labels = []

print("\nExtracting audio features...")

for index, row in metadata.iterrows():

    audio_path = audio_folder / row["filename"]

    try:
        audio_features = extract_mfcc(str(audio_path))
        features.append(audio_features)
        labels.append(row["category"])

    except Exception as error:
        print("Skipping file:", row["filename"], "|", error)

    if (index + 1) % 100 == 0:
        print(f"Processed {index + 1}/{len(metadata)} recordings")

X = np.asarray(features, dtype=np.float32)
labels = np.asarray(labels)

if len(X) == 0:
    raise RuntimeError(
        "No audio files were processed. Check the dataset download."
    )

# Convert category names into numeric IDs for PyTorch.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
class_names = label_encoder.classes_

print("\nFeature matrix shape:", X.shape)
print("Number of classes:", len(class_names))


# STEP 7: SPLIT THE DATA INTO TRAINING AND TESTING SETS
# The model learns from the training set and is evaluated
# on separate recordings that it has not trained on.
# The scaler is fitted only on the training data to prevent leakage.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

print("Training recordings:", len(X_train))
print("Testing recordings:", len(X_test))


# STEP 8: BUILD THE DEEP-LEARNING MODEL
# This MLP learns patterns from the extracted audio features.
# ReLU adds non-linearity, while dropout helps reduce overfitting.

class AudioClassifier(nn.Module):

    def __init__(self, input_size, number_of_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, number_of_classes)
        )

    def forward(self, x):
        return self.network(x)


model = AudioClassifier(
    input_size=X_train.shape[1],
    number_of_classes=len(class_names)
).to(device)

# Cross-entropy is suitable for multi-class classification.
criterion = nn.CrossEntropyLoss()

# Adam adjusts the model weights during training.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# STEP 9: TRAIN THE MODEL
# Each epoch passes the training data through the model once.
# The loss indicates how far the predictions are from the labels.

EPOCHS = 15
training_losses = []

print("\nStarting model training...")

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0.0

    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Clear gradients from the previous update.
        optimizer.zero_grad()

        # Predict the sound category.
        outputs = model(batch_features)

        # Calculate prediction error.
        loss = criterion(outputs, batch_labels)

        # Backpropagation calculates gradients.
        loss.backward()

        # Update model weights.
        optimizer.step()

        total_loss += loss.item() * len(batch_features)

    average_loss = total_loss / len(train_dataset)
    training_losses.append(average_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"- Training Loss: {average_loss:.4f}"
    )

print("\nTraining completed!")


# STEP 10: EVALUATE THE MODEL
# Accuracy measures the overall proportion of correct predictions.
# Macro precision, recall and F1 give equal importance to each class.

model.eval()

with torch.no_grad():
    predictions_output = model(X_test_tensor)
    probabilities = torch.softmax(predictions_output, dim=1)
    y_pred = probabilities.argmax(dim=1).cpu().numpy()

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, average="macro", zero_division=0
)
recall = recall_score(
    y_test, y_pred, average="macro", zero_division=0
)
f1 = f1_score(
    y_test, y_pred, average="macro", zero_division=0
)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-Score"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

print("\nEvaluation Metrics:")
print(results.to_string(index=False))

print("\nDetailed Classification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0
    )
)


# STEP 11: VISUALISE TRAINING LOSS
# A decreasing loss generally indicates that the model
# is learning from the training data.

plt.figure(figsize=(8, 4))
plt.plot(
    range(1, EPOCHS + 1),
    training_losses,
    marker="o"
)
plt.title("Training Loss Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True)
plt.tight_layout()
plt.show()


# STEP 12: DISPLAY THE CONFUSION MATRIX
# The confusion matrix shows which sound categories
# are correctly classified and which are confused.

matrix = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(class_names))
)

plt.figure(figsize=(15, 12))
sns.heatmap(
    matrix,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("ESC-50 Audio Classification Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


# STEP 13: PREDICT A SOUND FROM A NEW AUDIO FILE
# Put your audio file in the notebook folder and set AUDIO_PATH.
# Example: AUDIO_PATH = "my_sound.wav"
# The file should contain a sound the classifier can attempt to classify.

def predict_sound(audio_path):

    sample_features = extract_mfcc(audio_path).reshape(1, -1)
    sample_features = scaler.transform(sample_features).astype(
        np.float32
    )

    sample_tensor = torch.tensor(
        sample_features,
        dtype=torch.float32
    ).to(device)

    model.eval()

    with torch.no_grad():
        output = model(sample_tensor)
        probs = torch.softmax(output, dim=1)[0]

    predicted_id = int(torch.argmax(probs).item())
    predicted_label = class_names[predicted_id]
    confidence = float(probs[predicted_id].item())

    print("\nPredicted sound:", predicted_label)
    print(f"Model confidence: {confidence:.2%}")

    return predicted_label, confidence


AUDIO_PATH = ""  # Example: "my_sound.wav"

predicted_label = None

if AUDIO_PATH and os.path.exists(AUDIO_PATH):

    predicted_label, confidence = predict_sound(AUDIO_PATH)

    # Listen to the uploaded audio.
    signal, sr = librosa.load(AUDIO_PATH, sr=22050)
    display(Audio(signal, rate=sr))

else:
    print(
        "\nNo custom audio file selected. "
        "Set AUDIO_PATH to test your own recording."
    )

    # Demonstrate the model with one held-out test recording.
    demo_index = 0
    predicted_id = int(y_pred[demo_index])

    predicted_label = class_names[predicted_id]

    print("Example actual category:", class_names[y_test[demo_index]])
    print("Example predicted category:", predicted_label)
    print(
        f"Example confidence: "
        f"{probabilities[demo_index, predicted_id].item():.2%}"
    )


# STEP 14: SPEECH-TO-TEXT USING WHISPER
# This section is optional and requires a separate speech recording.
# ESC-50 contains environmental sounds, so it is not the dataset
# used to evaluate speech recognition.
#
# Put a speech recording named "speech.wav" in the notebook folder,
# or change SPEECH_PATH to point to your file.
# Whisper Tiny downloads its model the first time it is used.

SPEECH_PATH = "speech.wav"
transcript_text = ""

if os.path.exists(SPEECH_PATH):

    from transformers import pipeline

    print("\nLoading Whisper Tiny...")

    speech_recognizer = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-tiny",
        device=0 if device.type == "cuda" else -1
    )

    speech_result = speech_recognizer(SPEECH_PATH)
    transcript_text = speech_result.get("text", "").strip()

    print("\nRecognised speech:")
    print(transcript_text)

else:
    print(
        "\nSTT skipped: no speech.wav file was found. "
        "Add a speech recording to run Whisper."
    )


# STEP 15: TEXT-TO-SPEECH USING gTTS
# gTTS converts text into an audio file.
# By default, the program speaks the predicted sound category.
# Change TTS_MODE to "transcript" to speak Whisper's transcript instead.
# An internet connection is required for gTTS.

from gtts import gTTS

TTS_MODE = "label"

if TTS_MODE == "transcript" and transcript_text:

    text_to_speak = "The recognised speech is: " + transcript_text

elif predicted_label is not None:

    text_to_speak = (
        "The predicted environmental sound is " + predicted_label
    )

else:

    text_to_speak = "Audio classification has been completed."

print("\nText-to-speech output:")
print(text_to_speak)

try:
    tts = gTTS(text=text_to_speak, lang="en")
    tts.save("tts_output.mp3")

    print("Speech saved as tts_output.mp3")
    display(Audio(filename="tts_output.mp3"))

except Exception as error:
    print("TTS could not run. Check your internet connection.")
    print("Details:", error)


# STEP 16: SAVE THE MODEL AND RESULTS
# Saving these files lets you keep the trained model and metrics.
# The scaler and class names are needed to interpret future predictions.

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_size": X_train.shape[1],
        "number_of_classes": len(class_names),
        "class_names": list(class_names)
    },
    "esc50_audio_classifier.pt"
)

joblib.dump(scaler, "esc50_scaler.joblib")
joblib.dump(label_encoder, "esc50_label_encoder.joblib")
results.to_csv("audio_classification_metrics.csv", index=False)

print("\nFiles saved successfully:")
print("- esc50_audio_classifier.pt")
print("- esc50_scaler.joblib")
print("- esc50_label_encoder.joblib")
print("- audio_classification_metrics.csv")


# STEP 17: FINAL SUMMARY
# Use the actual metrics printed above in your assignment report.

print("\n========== PROJECT SUMMARY ==========")
print("Dataset: ESC-50 Environmental Sound Classification")
print("Model: PyTorch Multi-Layer Perceptron")
print("Audio features: MFCC mean and standard deviation")
print(f"Number of classes: {len(class_names)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1-Score: {f1:.4f}")
print("Speech-to-Text: Whisper Tiny (optional)")
print("Text-to-Speech: Google Text-to-Speech (gTTS)")